# ProtoNet v3 — Few-Shot Liver Tumor Classification

**Stage 2** of the two-stage pipeline: classify liver tumors as **Benign** or **Malignant** using Prototypical Networks with a ResNet-34 backbone.

### Improvements over v2

| # | Technique | Purpose |
|---|---|---|
| 1 | Targeted data augmentation | Reduce overfitting on small training set |
| 2 | Projection head with Dropout | Learn better embeddings, regularize |
| 3 | Weight decay (L2) | Penalize large weights |
| 4 | Label smoothing | Prevent overconfident predictions |
| 5 | Cosine annealing warm restarts | Smoother LR schedule, escape local minima |
| 6 | Early stopping | Halt training when validation plateaus |
| 7 | Optimal threshold tuning | Maximize recall for malignant class |

## 1 &ensp; Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os, random, time
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve,
    classification_report, roc_curve,
)
import matplotlib.pyplot as plt

%matplotlib inline

## 2 &ensp; Configuration

In [ ]:
CONFIG = {
    # ── Reproducibility ──
    "seed": 42,

    # ── Data ──
    "benign_dir":     "../data/train/benign",
    "malignant_dir":  "../data/train/malignant",
    "test_size":       0.3,
    "val_test_split":  0.5,

    # ── Model ──
    "backbone":      "resnet34",
    "dropout_rate":   0.15,    # light dropout to preserve embedding quality

    # ── Episodic training ──
    "num_episodes":  800,
    "K_shot":         30,
    "Q_query":        30,
    "N_way":           2,

    # ── Optimiser ──
    "lr":             5e-5,
    "weight_decay":   1e-4,
    "label_smoothing": 0.05,   # gentle smoothing to preserve calibration

    # ── LR scheduler (CosineAnnealingWarmRestarts) ──
    "T_0":   50,
    "T_mult": 2,
    "eta_min": 1e-7,

    # ── Early stopping ──
    "patience":   30,
    "min_delta":  0.005,

    # ── Logging ──
    "val_interval": 10,

    # ── Output ──
    "model_path": "../models/v3_protonet_liver_tumor_resnet34.pth",
}

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".gif"}


def set_seed(seed: int) -> None:
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 3 &ensp; Data Augmentation

Moderate augmentation — strong enough to regularize, light enough to
preserve medical image features. Only spatially mild transforms and
light color jitter are used.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## 4 &ensp; Dataset & Data Loading

In [ ]:
class LiverTumorDataset:
    """Simple map-style dataset for liver tumor images."""

    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self._idx = {}
        for i, l in enumerate(labels):
            self._idx.setdefault(l, []).append(i)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[i]

    def class_indices(self, c):
        """Return indices belonging to class *c*."""
        return self._idx.get(c, [])


def load_data(cfg):
    """Load images and split into train / val / test."""
    benign = sorted(p for p in Path(cfg["benign_dir"]).iterdir()
                    if p.suffix.lower() in IMG_EXTS)
    malignant = sorted(p for p in Path(cfg["malignant_dir"]).iterdir()
                       if p.suffix.lower() in IMG_EXTS)

    paths  = [str(p) for p in benign + malignant]
    labels = [0] * len(benign) + [1] * len(malignant)

    print(f"Total images: {len(paths)}  "
          f"(benign={len(benign)}, malignant={len(malignant)})")

    tr_p, tmp_p, tr_l, tmp_l = train_test_split(
        paths, labels,
        test_size=cfg["test_size"], random_state=cfg["seed"],
        stratify=labels)

    va_p, te_p, va_l, te_l = train_test_split(
        tmp_p, tmp_l,
        test_size=cfg["val_test_split"], random_state=cfg["seed"],
        stratify=tmp_l)

    print(f"Split  \u2192  train={len(tr_p)}  val={len(va_p)}  test={len(te_p)}")

    train_ds      = LiverTumorDataset(tr_p, tr_l, train_transform)
    train_ds_eval = LiverTumorDataset(tr_p, tr_l, eval_transform)
    val_ds        = LiverTumorDataset(va_p, va_l, eval_transform)
    test_ds       = LiverTumorDataset(te_p, te_l, eval_transform)
    return train_ds, train_ds_eval, val_ds, test_ds


train_ds, train_ds_eval, val_ds, test_ds = load_data(CONFIG)

## 5 &ensp; Model

A **Prototypical Network** with a lightweight projection head.

```
ResNet-34 encoder  \u2192  [512-d]  \u2192  Linear \u2192 ReLU \u2192 Dropout(0.15) \u2192 Linear  \u2192  [512-d embedding]
```

In [ ]:
class ProtoNet(nn.Module):
    """Prototypical Network with a projection head.

    The projection head improves embedding quality and the
    dropout layer acts as an additional regulariser.
    """

    def __init__(self, backbone="resnet34", dropout=0.15):
        super().__init__()
        self.encoder = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        dim = self.encoder.fc.in_features  # 512
        self.encoder.fc = nn.Identity()

        self.projection = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return self.projection(self.encoder(x))


model = ProtoNet(dropout=CONFIG["dropout_rate"]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 6 &ensp; Utilities

In [ ]:
class EarlyStopping:
    """Stop training when the monitored metric stops improving."""

    def __init__(self, patience=30, min_delta=0.005):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best = None
        self.triggered = False

    def step(self, score):
        if self.best is None or score > self.best + self.min_delta:
            self.best = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True


def compute_prototypes(embeddings, labels, n_classes):
    """Average embeddings per class to obtain prototypes."""
    return torch.stack([embeddings[labels == c].mean(0)
                        for c in range(n_classes)])


def sq_euclidean(a, b):
    """Squared Euclidean distance matrix.  a:(N,D)  b:(M,D) -> (N,M)"""
    return torch.cdist(a.float(), b.float(), p=2).pow(2)


def sample_episode(ds, K, Q, device):
    """Sample one ProtoNet episode (K support + Q query per class)."""
    idx0 = ds.class_indices(0)
    idx1 = ds.class_indices(1)
    cap = min(len(idx0), len(idx1))
    k = min(K, cap // 2)
    q = min(Q, cap - k)

    s0 = random.sample(idx0, k + q)
    s1 = random.sample(idx1, k + q)

    def _load(indices):
        imgs, labs = zip(*[ds[i] for i in indices])
        return (torch.stack(imgs).to(device),
                torch.tensor(labs, dtype=torch.long, device=device))

    sup_imgs, sup_labs = _load(s0[:k] + s1[:k])
    qry_imgs, qry_labs = _load(s0[k:] + s1[k:])
    return sup_imgs, sup_labs, qry_imgs, qry_labs

## 7 &ensp; Training & Evaluation

In [ ]:
def train_episode(model, sup_imgs, sup_labs, qry_imgs, qry_labs,
                  optimiser, criterion, scaler, n_way):
    """Run one training episode and return loss, accuracy, F1."""
    model.train()
    optimiser.zero_grad()
    with autocast(enabled=(DEVICE == "cuda")):
        protos = compute_prototypes(model(sup_imgs), sup_labs, n_way)
        logits = -sq_euclidean(model(qry_imgs), protos)
        loss = criterion(logits, qry_labs)
    scaler.scale(loss).backward()
    scaler.step(optimiser)
    scaler.update()

    y_true = qry_labs.cpu().numpy()
    y_pred = logits.argmax(1).detach().cpu().numpy()
    return (loss.item(),
            accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average="weighted", zero_division=0))


@torch.no_grad()
def full_evaluate(model, support_ds, eval_ds, n_way, device, bs=64):
    """Non-episodic evaluation.

    Prototypes are computed from the **entire** support set, then
    every sample in *eval_ds* is classified. This gives stable,
    reproducible metrics unlike episodic evaluation.

    Returns
    -------
    acc, prec, rec, f1, auc, y_true, y_pred, y_prob
    """
    model.eval()

    # ── Compute prototypes from full support set ──
    embs, labs = [], []
    for i in range(0, len(support_ds), bs):
        batch_imgs, batch_labs = zip(*[support_ds[j]
                                       for j in range(i, min(i + bs, len(support_ds)))])
        with autocast(enabled=(device == "cuda")):
            embs.append(model(torch.stack(batch_imgs).to(device)).cpu())
        labs.extend(batch_labs)
    protos = compute_prototypes(
        torch.cat(embs), torch.tensor(labs, dtype=torch.long), n_way
    ).to(device)

    # ── Classify every sample in eval set ──
    y_true, y_pred, y_prob = [], [], []
    for i in range(len(eval_ds)):
        img, lab = eval_ds[i]
        with autocast(enabled=(device == "cuda")):
            emb = model(img.unsqueeze(0).to(device))
        logits = -sq_euclidean(emb, protos)
        prob = F.softmax(logits, dim=1)
        y_true.append(lab)
        y_pred.append(logits.argmax(1).item())
        y_prob.append(prob[0, 1].item())

    y_true, y_pred, y_prob = map(np.array, (y_true, y_pred, y_prob))
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.0
    return acc, prec, rec, f1, auc, y_true, y_pred, y_prob

## 8 &ensp; Training Loop

In [ ]:
os.makedirs("../results", exist_ok=True)
os.makedirs("../models", exist_ok=True)

optimiser = torch.optim.Adam(model.parameters(),
                             lr=CONFIG["lr"],
                             weight_decay=CONFIG["weight_decay"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimiser, T_0=CONFIG["T_0"], T_mult=CONFIG["T_mult"],
    eta_min=CONFIG["eta_min"])

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
scaler    = GradScaler(enabled=(DEVICE == "cuda"))
stopper   = EarlyStopping(CONFIG["patience"], CONFIG["min_delta"])

hist = dict(tr_loss=[], tr_acc=[], tr_f1=[],
            va_acc=[], va_f1=[], va_rec=[], lr=[], ep=[])

best_f1, best_ep = 0.0, 0
t0 = time.time()

N = CONFIG["num_episodes"]
print(f"{'='*72}")
print(f"Episodes={N}  K={CONFIG['K_shot']}  Q={CONFIG['Q_query']}  "
      f"N-way={CONFIG['N_way']}")
print(f"lr={CONFIG['lr']}  wd={CONFIG['weight_decay']}  "
      f"label_smooth={CONFIG['label_smoothing']}  "
      f"dropout={CONFIG['dropout_rate']}")
print(f"patience={CONFIG['patience']}  T_0={CONFIG['T_0']}  "
      f"T_mult={CONFIG['T_mult']}")
print(f"{'='*72}\n")

for ep in range(1, N + 1):
    si, sl, qi, ql = sample_episode(
        train_ds, CONFIG["K_shot"], CONFIG["Q_query"], DEVICE)
    loss, acc, f1 = train_episode(
        model, si, sl, qi, ql,
        optimiser, criterion, scaler, CONFIG["N_way"])
    scheduler.step()

    hist["tr_loss"].append(loss)
    hist["tr_acc"].append(acc)
    hist["tr_f1"].append(f1)

    if ep % CONFIG["val_interval"] == 0 or ep == 1:
        va, vp, vr, vf, vauc, *_ = full_evaluate(
            model, train_ds_eval, val_ds, CONFIG["N_way"], DEVICE)
        hist["va_acc"].append(va)
        hist["va_f1"].append(vf)
        hist["va_rec"].append(vr)
        hist["ep"].append(ep)
        hist["lr"].append(optimiser.param_groups[0]["lr"])

        tag = ""
        if vf > best_f1:
            best_f1, best_ep = vf, ep
            torch.save(model.state_dict(), CONFIG["model_path"])
            tag = "  \u2605 saved"

        print(f"ep {ep:4d}/{N} | loss {loss:.4f} | "
              f"tr_acc {acc:.3f}  tr_f1 {f1:.3f} | "
              f"va_acc {va:.3f}  va_f1 {vf:.3f}  va_rec {vr:.3f} | "
              f"lr {optimiser.param_groups[0]['lr']:.1e} | "
              f"{time.time()-t0:.0f}s{tag}")

        stopper.step(vf)
        if stopper.triggered:
            print(f"\nEarly stopping after {CONFIG['patience']} "
                  f"episodes without improvement.")
            break

print(f"\nTraining finished in {time.time()-t0:.0f}s")
print(f"Best val F1 = {best_f1:.4f} at episode {best_ep}")

model.load_state_dict(torch.load(
    CONFIG["model_path"], map_location=DEVICE, weights_only=True))
print("Best model loaded.")

## 9 &ensp; Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ProtoNet v3 \u2014 Training Metrics",
             fontsize=16, fontweight="bold")

ex = range(1, len(hist["tr_loss"]) + 1)
ev = hist["ep"]

ax = axes[0, 0]
ax.plot(ex, hist["tr_loss"], alpha=0.25, label="raw")
w = min(20, len(hist["tr_loss"]))
if w > 1:
    sm = np.convolve(hist["tr_loss"], np.ones(w)/w, "valid")
    ax.plot(range(w, len(sm)+w), sm, lw=2, label="smoothed")
ax.set(xlabel="Episode", ylabel="Loss", title="Training Loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(ex, hist["tr_acc"], alpha=0.25, label="train")
ax.plot(ev, hist["va_acc"], "o-", ms=3, lw=2, label="val")
ax.set(xlabel="Episode", ylabel="Accuracy", title="Accuracy",
       ylim=(0, 1.05))
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(ex, hist["tr_f1"], alpha=0.25, label="train")
ax.plot(ev, hist["va_f1"], "o-", ms=3, lw=2, label="val")
ax.set(xlabel="Episode", ylabel="F1", title="F1 Score",
       ylim=(0, 1.05))
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(ev, hist["lr"], "g-", lw=2)
ax.set(xlabel="Episode", ylabel="LR", title="Learning Rate")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/v3_training_metrics.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 10 &ensp; Test-Set Evaluation

In [ ]:
te_acc, te_prec, te_rec, te_f1, te_auc, y_true, y_pred, y_prob = \
    full_evaluate(model, train_ds_eval, test_ds, CONFIG["N_way"], DEVICE)

print(f"Acc={te_acc:.4f}  Prec={te_prec:.4f}  "
      f"Rec={te_rec:.4f}  F1={te_f1:.4f}  AUC={te_auc:.4f}\n")
print(classification_report(y_true, y_pred,
                            target_names=["Benign", "Malignant"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("ProtoNet v3 \u2014 Test Evaluation",
             fontsize=14, fontweight="bold")

cm = confusion_matrix(y_true, y_pred)
im = ax1.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(cm[i, j]), ha="center", va="center",
                 fontsize=20,
                 color="white" if cm[i, j] > cm.max()/2 else "black")
ax1.set(xticks=[0,1], yticks=[0,1],
        xticklabels=["Benign","Malignant"],
        yticklabels=["Benign","Malignant"],
        xlabel="Predicted", ylabel="True", title="Confusion Matrix")
fig.colorbar(im, ax=ax1)

fpr, tpr, _ = roc_curve(y_true, y_prob)
ax2.plot(fpr, tpr, lw=2, label=f"AUC = {te_auc:.4f}")
ax2.plot([0,1],[0,1], "r--", alpha=0.4)
ax2.set(xlabel="FPR", ylabel="TPR", title="ROC Curve")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/v3_test_evaluation.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 11 &ensp; Threshold Tuning

In a medical setting, missing a malignant case (false negative) is
far more costly than a false positive.  We search for the decision
threshold that achieves **recall \u2265 90 %** for the malignant class
while maximising precision.

In [ ]:
TARGET_RECALL = 0.90

precs, recs, threshs = precision_recall_curve(y_true, y_prob)

mask = recs[:-1] >= TARGET_RECALL
if mask.any():
    ix = np.where(mask)[0][precs[:-1][mask].argmax()]
    opt_t  = threshs[ix]
    opt_p  = precs[ix]
    opt_r  = recs[ix]
    y_opt  = (y_prob >= opt_t).astype(int)

    print(f"Optimal threshold : {opt_t:.4f}")
    print(f"Precision         : {opt_p:.4f}")
    print(f"Recall            : {opt_r:.4f}")
    print(f"Accuracy          : {accuracy_score(y_true, y_opt):.4f}")
    print(f"F1 (weighted)     : "
          f"{f1_score(y_true, y_opt, average='weighted', zero_division=0):.4f}")
    print()
    print(classification_report(y_true, y_opt,
                                target_names=["Benign", "Malignant"]))
else:
    opt_t, opt_p, opt_r = 0.5, 0, 0
    print(f"Could not find a threshold with recall >= {TARGET_RECALL:.0%}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recs[:-1], precs[:-1], lw=2)
ax.axvline(TARGET_RECALL, color="r", ls="--",
           label=f"target recall = {TARGET_RECALL:.0%}")
if mask.any():
    ax.plot(opt_r, opt_p, "ro", ms=10,
            label=f"optimal (P={opt_p:.3f}, R={opt_r:.3f})")
ax.set(xlabel="Recall (malignant)", ylabel="Precision (malignant)",
       title="Precision-Recall Trade-off")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../results/v3_threshold_tuning.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 12 &ensp; Summary

In [ ]:
print(f"{'='*60}")
print(f"  Model saved to     : {CONFIG['model_path']}")
print(f"  Best val F1        : {best_f1:.4f}  (episode {best_ep})")
print(f"  Test  Acc / F1     : {te_acc:.4f} / {te_f1:.4f}")
print(f"  Test  AUC          : {te_auc:.4f}")
print(f"  Optimal threshold  : {opt_t:.4f}")
print(f"{'='*60}")